# CER is tone-blind — a demonstration (Paper-1 methods motivation)

The standard TTS intelligibility metric, **CER**, cannot see lexical tone — so a tonal-TTS paper needs a
*tone* metric. This is the project's premise; it can't be cited, so we **prove it** here, cheaply.

**Method (airtight, with a control).** Take real, correctly-toned Yoruba clips. Make three versions of each:
- **orig** — real audio (correct tone)
- **roundtrip** — `psola_roundtrip`: PSOLA resynthesis with **no pitch edit** (same artifact, tone UNCHANGED) → the **control**
- **flat** — `psola_flatten`: F0 flattened to its mean (monotone — **tone destroyed**, words intact)

Score every version with **CER** (MMS-1b-all `yor`) and **`tone_i2`** (the project's F0-absolute tone meter).
Expected, and the whole point:

| | CER | tone_i2 |
|---|---|---|
| orig | low | high |
| roundtrip (control) | ≈ orig | ≈ orig |
| **flat** | **≈ orig (tone-blind!)** | **→ chance 0.33 (collapses)** |

`ΔCER(flat−orig) ≈ 0` while `Δtone_i2 ≈ −0.3`. The roundtrip control (artifact present, tone intact) keeps
**both** unchanged, proving the `tone_i2` collapse is caused by **tone removal**, not the resynthesis artifact.

**GPU: T4/L4** (gate scoring only; PSOLA is CPU). `hf-xet` removed.

## 1. Install + restart

In [ ]:
%pip install --quiet "numpy<2"
%pip install --quiet boto3 soundfile soxr librosa speechbrain "swift-f0" pyworld praat-parselmouth transformers safetensors "huggingface_hub>=0.24.0" tqdm matplotlib
%pip uninstall -y hf-xet
import os
print("Installs done. RESTARTING so numpy<2 takes effect — run from the NEXT cell.", flush=True)
os._exit(0)

In [ ]:
import numpy as np
assert np.__version__.startswith("1."), "numpy<2 pin did not take — re-run install + restart"
import parselmouth; print("numpy",np.__version__,"| parselmouth",getattr(parselmouth,"VERSION","?"))

## 2. Clone `mosesdaudu001/tone-on-a-budget` (approachB: tone_oracle PSOLA + tone_f0_abs + grpo_reward)

In [ ]:
# tone-metric package (public) — replaces the old git clone + sys.path of tone_metric/
%pip install --quiet --no-deps "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
import os, subprocess, sys, shutil, importlib
importlib.invalidate_caches()
import tone_metric
from tone_metric import tone_oracle as toq   # psola_flatten / psola_roundtrip
from tone_metric import tone_f0_abs as f0a   # I2 tone meter
from tone_metric.grpo_reward import RewardModels   # CER
CODE_DIR = os.path.dirname(tone_metric.__file__)   # minimal_pairs_draft.json ships as package data
print("tone_metric", tone_metric.__version__, "from", CODE_DIR)


## 3. Secrets + S3 + gate stack (CER + I2) + F0 calibration

In [ ]:
import os, getpass, boto3, torch, numpy as np, json
def _secret(k):
    try:
        from google.colab import userdata
        v=userdata.get(k)
        if v: return v
    except Exception: pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")
os.environ["AWS_ACCESS_KEY_ID"]=_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"]=_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]=os.environ.get("AWS_DEFAULT_REGION","us-east-1")
HF_TOKEN=_secret("HF_TOKEN"); os.environ["HF_TOKEN"]=HF_TOKEN or ""
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN)
os.environ["HF_HUB_DISABLE_XET"]="1"; os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="0"
try:
    import huggingface_hub.constants as _hc; _hc.HF_HUB_DISABLE_XET=True
except Exception: pass
BUCKET="codec-audio-data"; s3=boto3.client("s3",region_name=os.environ["AWS_DEFAULT_REGION"]); s3.head_bucket(Bucket=BUCKET)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"; SR=24000
BIBLE_MANIFEST="tts_data/yoruba_gold/s1_train.v2.jsonl"
F0CAL_KEY="tts_data/yoruba/tone_v2/f0_abs_calibration.v1.json"
OUT_PREFIX="tts_data/yoruba/cer_tone_blind"
WORK="/content/cerblind" if os.path.isdir("/content") else "/tmp/cerblind"; os.makedirs(WORK,exist_ok=True)
rm=RewardModels(device=DEVICE)
s3.download_file(BUCKET,F0CAL_KEY,f"{WORK}/f0cal.json"); F0CAL=json.load(open(f"{WORK}/f0cal.json"))
I2_TH,I2_TL=F0CAL["theta_h"],F0CAL["theta_l"]; I2_MODE,I2_LATE=F0CAL.get("mode","blind"),F0CAL.get("late_frac",0.5)
print("gate ready | DEVICE",DEVICE,"| theta_h",I2_TH,"theta_l",I2_TL)

## 4. Pull real, correctly-toned Yoruba clips (bible + SLR86)

In [ ]:
import io, json, soundfile as sf, numpy as np
N_CLIPS=12
clips=[]; seen_spk={}
for raw in io.TextIOWrapper(s3.get_object(Bucket=BUCKET,Key=BIBLE_MANIFEST)["Body"],encoding="utf-8"):
    r=json.loads(raw)
    if not (3.0<=float(r.get("duration_sec",0))<=9.0): continue
    spk=str(r.get("speaker_id","?")); src=r.get("source","?")
    seen_spk[src]=seen_spk.get(src,0)+1
    if seen_spk[src]>N_CLIPS: continue           # spread across sources/speakers
    lp=f"{WORK}/clip_{r['id']}.wav"; s3.download_file(BUCKET,r["audio_s3_key"],lp)
    w,sr=sf.read(lp,dtype="float32"); w=w.mean(-1) if w.ndim>1 else w
    if sr!=SR:                                    # resample defensively
        import soxr; w=soxr.resample(w,sr,SR).astype("float32"); sr=SR
    clips.append(dict(id=r["id"],text=r["text"],source=src,wav=w))
    if len(clips)>=N_CLIPS: break
assert len(clips)>=4, "too few clips"
print("real Yoruba clips:",len(clips),"| sources:",{c['source'] for c in clips})

## 5. Make orig / roundtrip / flat, and score each with CER + tone_i2

In [ ]:
from collections import defaultdict
from tqdm.auto import tqdm
import numpy as np
def _bal(pairs):
    if not pairs: return float("nan")
    tot,cor=defaultdict(int),defaultdict(int)
    for pp,tt in pairs: tot[tt]+=1; cor[tt]+=int(pp==tt)
    rs=[cor[c]/tot[c] for c in tot if tot[c]>0]; return float(np.mean(rs)) if rs else float("nan")
def cer_tone(wav,text):
    logits,n16=rm.asr_logits(wav,SR)
    cer=RewardModels.cer(text,rm.decode_logits(logits))
    i2=f0a.f0_abs_score(wav,SR,text,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,emissions=logits,n16=n16,
                        theta_h=I2_TH,theta_l=I2_TL,mode=I2_MODE,mid_ref=None,late_frac=I2_LATE)
    pairs=[(pp,tt) for pp,tt in zip(i2["pred"],i2["target"]) if pp is not None]
    return cer,_bal(pairs),pairs

COND={"orig":lambda w:w,
      "roundtrip":lambda w:toq.psola_roundtrip(w,SR),   # artifact control — tone UNCHANGED
      "flat":lambda w:toq.psola_flatten(w,SR)}           # monotone — tone DESTROYED
rows=[]; variants={}
for c in tqdm(clips,desc="clips"):
    rec={"id":c["id"],"text":c["text"]}
    for name,fn in COND.items():
        try: w=np.asarray(fn(c["wav"]),dtype="float32")
        except Exception as e: print("  PSOLA fail",c["id"],name,"->",e); w=c["wav"]
        cer,t2,pairs=cer_tone(w,c["text"])
        rec[name]={"cer":cer,"tone_i2":t2,"pairs":pairs}
        variants.setdefault(c["id"],{})[name]=w
    rows.append(rec)

def agg(name):
    cers=[r[name]["cer"] for r in rows if r[name]["cer"]==r[name]["cer"]]
    allp=[p for r in rows for p in r[name]["pairs"]]
    t2s=[r[name]["tone_i2"] for r in rows if r[name]["tone_i2"]==r[name]["tone_i2"]]
    return dict(cer_mean=float(np.mean(cers)),cer_std=float(np.std(cers)),
                tone_i2_pooled=_bal(allp),tone_i2_mean=float(np.mean(t2s)),tone_i2_std=float(np.std(t2s)))
A={n:agg(n) for n in COND}
print(f"{'condition':12} {'CER':>14} {'tone_i2(pooled)':>16} {'tone_i2(clip mean±sd)':>22}")
for n in ["orig","roundtrip","flat"]:
    a=A[n]; print(f"{n:12} {a['cer_mean']:>7.3f}±{a['cer_std']:.3f} {a['tone_i2_pooled']:>16.3f} "
                  f"{a['tone_i2_mean']:>14.3f}±{a['tone_i2_std']:.3f}")
dCER=A["flat"]["cer_mean"]-A["orig"]["cer_mean"]; dT2=A["flat"]["tone_i2_pooled"]-A["orig"]["tone_i2_pooled"]
rtCER=A["roundtrip"]["cer_mean"]-A["orig"]["cer_mean"]; rtT2=A["roundtrip"]["tone_i2_pooled"]-A["orig"]["tone_i2_pooled"]
print(f"\nFLAT vs ORIG:  ΔCER = {dCER:+.3f}   Δtone_i2 = {dT2:+.3f}   (chance tone_i2 = 0.333)")
print(f"CONTROL roundtrip vs ORIG: ΔCER = {rtCER:+.3f}  Δtone_i2 = {rtT2:+.3f}  (both ~0 => artifact is harmless)")
verdict = (abs(dCER)<0.08) and (dT2 < -0.12) and (abs(rtT2)<0.12)
print("\n=>", "DEMONSTRATED: CER is tone-blind (ΔCER~0 when tone destroyed) while tone_i2 collapses; "
      "roundtrip control rules out the artifact." if verdict else
      "INSPECT: numbers did not cleanly separate — check per-clip table / audio below.")

**5b. Dose-response — tone_i2 tracks the degree of tone error (CER doesn't)**

The binary orig/flat test shows the meter notices tone is gone. Stronger: resynthesize each clip with its F0 excursions scaled about the mean — new_f0 = mean + factor*(f0 - mean) — sweeping factor = 1.0 -> 0.0 (1.0 = full tone, 0.0 = monotone). A good tone meter should degrade monotonically as tone is compressed, while CER stays flat. (factor=0 is exactly psola_flatten.)

In [ ]:

import numpy as np, matplotlib.pyplot as plt, json, datetime, soundfile as sf
import IPython.display as ipd
from parselmouth.praat import call
from tqdm.auto import tqdm
_F0MIN = getattr(toq, "F0MIN", 65); _F0MAX = getattr(toq, "F0MAX", 400)

def psola_scale_excursion(wav, factor):
    """new_f0 = mean + factor*(f0 - mean): factor=1 -> original tone, factor=0 -> flat (== psola_flatten)."""
    snd = toq._snd(np.asarray(wav, dtype="float32"), SR)
    manip = call(snd, "To Manipulation", 0.01, _F0MIN, _F0MAX)
    ptier = call(manip, "Extract pitch tier")
    n = int(call(ptier, "Get number of points"))
    if n == 0:
        return np.asarray(call(manip, "Get resynthesis (overlap-add)").values).reshape(-1).astype("float32")
    times = [call(ptier, "Get time from index", i) for i in range(1, n + 1)]
    vals  = [call(ptier, "Get value at index", i) for i in range(1, n + 1)]
    mean_hz = sum(vals) / len(vals)
    for i in range(n, 0, -1): call(ptier, "Remove point", i)
    for t, v in zip(times, vals): call(ptier, "Add point", float(t), float(mean_hz + factor * (v - mean_hz)))
    call([manip, ptier], "Replace pitch tier")
    return np.asarray(call(manip, "Get resynthesis (overlap-add)").values).reshape(-1).astype("float32")

FACTORS = [1.0, 0.75, 0.5, 0.25, 0.0]
dose = []
for fac in FACTORS:
    cers, allp = [], []
    for c in tqdm(clips, desc=f"tone factor {fac:.2f}"):
        try: w = psola_scale_excursion(c["wav"], fac)
        except Exception as e: print("  scale fail", c["id"], "->", e); continue
        cer, t2, pairs = cer_tone(w, c["text"]); cers.append(cer); allp += pairs
    dose.append(dict(factor=fac, cer=float(np.mean(cers)), tone_i2=_bal(allp)))

print(f"{'tone factor':>11} {'CER':>7} {'tone_i2':>8}")
for d in dose: print(f"{d['factor']:>11.2f} {d['cer']:>7.3f} {d['tone_i2']:>8.3f}")
t2s = [d["tone_i2"] for d in dose]; cers_all = [d["cer"] for d in dose]
mono = all(t2s[i] >= t2s[i + 1] - 0.03 for i in range(len(t2s) - 1))
rho = float(np.corrcoef(FACTORS, t2s)[0, 1])
print(f"\ntone_i2 monotone in tone strength: {mono} | corr(factor, tone_i2) = {rho:+.2f}")
print(f"CER span: {min(cers_all):.3f}-{max(cers_all):.3f} (flat)  vs  tone_i2 span: {min(t2s):.3f}-{max(t2s):.3f}")
print("=>", "DOSE-RESPONSE CONFIRMED: tone_i2 tracks tone strength monotonically; CER does not." if (mono and rho > 0.8) else "INSPECT the curve.")

plt.figure(figsize=(6, 4))
plt.plot(FACTORS, t2s, "o-", color="#F58518", label="tone_i2")
plt.plot(FACTORS, cers_all, "s--", color="#4C78A8", label="CER")
plt.axhline(1/3, ls=":", c="grey", label="chance 0.333")
plt.xlabel("tone strength  (F0 excursion factor: 1 = full tone, 0 = monotone)"); plt.ylabel("score")
plt.title("Dose-response: tone_i2 tracks tone strength; CER is flat"); plt.gca().invert_xaxis()
plt.legend(); plt.tight_layout(); plt.savefig(f"{WORK}/dose_response.png", dpi=120)
s3.upload_file(f"{WORK}/dose_response.png", BUCKET, f"{OUT_PREFIX}/dose_response.png")
s3.put_object(Bucket=BUCKET, Key=f"{OUT_PREFIX}/dose_response.json",
              Body=json.dumps(dict(date=datetime.date.today().isoformat(), factors=FACTORS, dose=dose,
                   monotone=bool(mono), corr=rho), ensure_ascii=False, indent=2).encode())
plt.show()

print("\nintermediate (factor 0.5 - half tone), by ear:")
for c in clips[:2]:
    w = psola_scale_excursion(c["wav"], 0.5); loc = f"{WORK}/dose_{c['id']}_0p5.wav"; sf.write(loc, w, SR)
    s3.upload_file(loc, BUCKET, f"{OUT_PREFIX}/listen/{c['id']}_factor0.5.wav")
    print(" ", c["text"][:50]); ipd.display(ipd.Audio(loc, rate=SR))
print(f"\n-> s3://{BUCKET}/{OUT_PREFIX}/{{dose_response.png, dose_response.json}}")

## 6. Plot + persist

In [ ]:
import matplotlib.pyplot as plt, numpy as np, json, datetime
names=["orig","roundtrip","flat"]
fig,ax=plt.subplots(1,2,figsize=(9,3.6))
ax[0].bar(names,[A[n]["cer_mean"] for n in names],yerr=[A[n]["cer_std"] for n in names],color="#4C78A8")
ax[0].set_title("CER (lower=intelligible)"); ax[0].set_ylabel("CER")
ax[1].bar(names,[A[n]["tone_i2_pooled"] for n in names],color="#F58518")
ax[1].axhline(1/3,ls=":",c="grey",label="chance 0.333"); ax[1].set_title("tone_i2 (higher=tone present)")
ax[1].set_ylabel("tone_i2"); ax[1].legend()
for a in ax: a.set_ylim(bottom=0)
plt.suptitle("CER is blind to tone; tone_i2 is not (roundtrip = artifact control)")
plt.tight_layout(); plt.savefig(f"{WORK}/cer_tone_blind.png",dpi=120)
s3.upload_file(f"{WORK}/cer_tone_blind.png",BUCKET,f"{OUT_PREFIX}/cer_tone_blind.png"); plt.show()
out=dict(date=datetime.date.today().isoformat(),n_clips=len(clips),agg=A,
         deltas=dict(flat_dCER=dCER,flat_dtone_i2=dT2,roundtrip_dCER=rtCER,roundtrip_dtone_i2=rtT2),
         per_clip=[{"id":r["id"],**{n:{"cer":r[n]["cer"],"tone_i2":r[n]["tone_i2"]} for n in names}} for r in rows])
s3.put_object(Bucket=BUCKET,Key=f"{OUT_PREFIX}/cerblind.json",Body=json.dumps(out,ensure_ascii=False,indent=2).encode())
print("-> s3://%s/%s/{cer_tone_blind.png, cerblind.json}"%(BUCKET,OUT_PREFIX))

## 7. LISTEN — same words, tone gone (the proof by ear)
Play orig vs **flat** (monotone) vs roundtrip for a few clips: the **words stay the same / intelligible**
(CER flat) but the **tune is destroyed** in `flat` (tone_i2 collapsed). roundtrip should sound ~like orig.

In [ ]:
import IPython.display as ipd, soundfile as sf, os
_up=[]
for c in clips[:3]:
    print(f"\n=== {c['id']} — {c['text'][:60]} ===")
    for n in ["orig","roundtrip","flat"]:
        w=variants[c["id"]][n]; loc=f"{WORK}/listen_{c['id']}_{n}.wav"; sf.write(loc,w,SR)
        s3.upload_file(loc,BUCKET,f"{OUT_PREFIX}/listen/{c['id']}_{n}.wav"); _up.append(n)
        r=[x for x in rows if x['id']==c['id']][0][n]
        print(f"  {n:10} cer {r['cer']:.2f}  tone_i2 {r['tone_i2']:.2f}"); ipd.display(ipd.Audio(loc,rate=SR))
print(f"\nuploaded {len(_up)} wavs -> s3://{BUCKET}/{OUT_PREFIX}/listen/")

### Read-out
- **`flat` keeps CER but craters `tone_i2`** → CER is blind to tone; an intelligibility metric can't
  certify tonal correctness. This is the in-paper figure (no citation needed).
- **`roundtrip` keeps both** → the `tone_i2` collapse is caused by **tone removal**, not the PSOLA
  artifact (the standard reviewer objection, pre-empted).
- The ear confirms: in `flat` the words are the same but the tune is gone.